# 19 - Generate LLM Context from Well Logs

**Part B - Petrophys Curriculum**

This lesson turns LAS data into compact, auditable JSON context packages organized by
analysis category, well, formation, and lithology group. It uses only public `petrophys`
APIs and bundled sample data, so it runs without machine-specific paths.

The generated JSON is model-neutral: use it with any LLM client, retrieval system, or
review workflow. Nothing in this lesson calls a model or the network.

## Learning goals

- Build context for one interpreted well with `pp.build_context(result)` and save it
  with `pp.export_context(result, path)`.
- Generate a computed package organized by category, formation and lithology group from
  a folder of LAS files plus optional formation and lithology tables.
- Select one level of a ranked formation table (`rank="formation"`) so groups and their
  formations are not counted twice.
- Enforce byte budgets and inspect provenance, warnings and limitations.
- Build a model-neutral request without making a network call.

### Prerequisites

- Lesson 10 (`pp.interpret`, the step trace, skip reasons and warnings) and lesson 11
  (formation tops and `rank="formation"`).
- The base install. Context generation runs locally and writes JSON; no LLM account,
  API key or network access is needed.
- The Rw convention used throughout the course: for 30/6-5, Rw = 0.070 ohm.m measured
  at 230 degF (`rw=0.070, rw_meas_temp_f=230`). Without the temperature the library
  assumes 75 degF, which changes saturation strongly.

## Why this matters

A language model knows only the text you give it. Raw depth-by-depth curves waste its
context window and leave it to invent the interpretation; a free-text summary hides
the assumptions. A context package sits in between: every number comes from the same
traced workflow as `pp.interpret`, grouped by the question being asked (hydrocarbon,
saturation, permeability, ...), bounded in size, and accompanied by the inputs,
warnings and data gaps a reviewer needs to judge it. The same files serve a retrieval
index, a review checklist or a human reader.

## What the exporter produces

For each formation and lithology group, the package exporter writes separate views for
hydrocarbon, saturation, permeability, porosity, numerical statistics, and lithology
analysis. Each JSON file records the source well, formation interval, input logs,
computed logs, statistics, analysis results, data completeness, and bad-hole QC.

Raw depth-by-depth traces are intentionally excluded by default. This keeps context
smaller while retaining the statistics and provenance needed for interpretation and
review.

## 1. Context for one interpreted well

`pp.build_context(result)` turns an interpretation result into a JSON-safe dictionary
(schema `1.0`) and never reruns a calculation; `pp.export_context(result, path)` writes
the same content to a file and enforces a byte budget (1,000,000 UTF-8 bytes by
default). Pass the **result**, not the raw well: only the result carries the calculated
curves and the workflow provenance.

Reading 30/6-5 prints two `UserWarning`s: the core saturation curves `SWC` and `SOC` are
stored in percent although the file labels them `frac`, so the reader divides them by
100 and says so. They reappear whenever this file is read, including inside the package
export below.

In [1]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import shutil

import pandas as pd
import petrophys as pp

print("petrophys version:", pp.__version__)
well = pp.read(pp.sample("30_6-5"))
result = pp.interpret(well, rw=0.070, rw_meas_temp_f=230.0).raise_for_errors()  # Rw (ohm.m) at 230 degF
context = pp.build_context(result)
assert context["interpretation"] is not None
print("schema:", context["schema_version"], "| kind:", context["context_kind"])
print("top-level keys:", list(context))
with TemporaryDirectory() as directory:
    saved = pp.export_context(result, Path(directory) / "context.json")
    print("Computed well context:", saved.name, saved.stat().st_size, "UTF-8 bytes")

petrophys version: 3.0.0


[repository]/src/petrophys/preprocessing/standardization/curve_standardizer.py:393: UserWarning: SWC: inferred percent core saturation despite unit 'frac'; divided by 100 to v/v. Verify source units.
  standardize_core_saturation(new_curve, inplace=True)
[repository]/src/petrophys/preprocessing/standardization/curve_standardizer.py:393: UserWarning: SOC: inferred percent core saturation despite unit 'frac'; divided by 100 to v/v. Verify source units.
  standardize_core_saturation(new_curve, inplace=True)


schema: 1.0 | kind: well_summary
top-level keys: ['schema_version', 'petrophys_version', 'context_kind', 'well', 'interpretation', 'geology', 'include_raw_traces_in_context']
Computed well context: context.json 14634 UTF-8 bytes


## 2. Configure package paths once

A package export reads a **folder of LAS files**. Only the LAS is mandatory; this
example also supplies optional formation and lithology tables. We copy the bundled
30/6-5 LAS into a fresh `logs` folder inside a temporary run directory and write
everything below it; no private workbook or earlier context package is read. Supported
well parameters and core curves are read from the LAS when present.

In [2]:
WELL_NAME = "30_6-5"
run_directory = TemporaryDirectory(prefix="petrophys-context-demo-")
RUN_ROOT = Path(run_directory.name)
LOGS_DIR = RUN_ROOT / "logs"             # a folder of LAS files, as for your own data
OUTPUT_DIR = RUN_ROOT / "context"        # must not exist before export
FORMATION_SOURCE = Path(pp.sample("formations_30_6-5"))
LITHOLOGY_SOURCE = Path(pp.sample("lithology_30_6-5"))
MINIMUM_GROUP_THICKNESS = 2.0            # in the LAS depth unit: metres here
MAX_BYTES = 1_000_000                    # per JSON document

LOGS_DIR.mkdir()
shutil.copy(pp.sample(WELL_NAME), LOGS_DIR)

print("LAS folder :", sorted(path.name for path in LOGS_DIR.iterdir()))
print("Formations :", FORMATION_SOURCE.name)
print("Lithology  :", LITHOLOGY_SOURCE.name)

LAS folder : ['30_6-5.las']
Formations : formations_30_6-5.csv
Lithology  : lithology_30_6-5.csv
Parameters : well_parameters.csv -> {'lithology_domain': 'auto', 'rw': 0.07, 'rw_meas_temp_f': 230.0, 'gr_min': 20.0, 'gr_max': 100.0}


For your own data, replace the LAS path and optionally fill the packaged formation or
lithology templates. CSV, TSV and Excel geology tables are accepted, including NPD/SODIR
`Well identifier` / `Surface` / `MD` headers. The [Context Export guide]
(https://petrophys.netlify.app/context) lists the sheet and depth metadata to supply.

## 3. Inspect the ranked formation table

The bundled 30/6-5 table declares metre depths, explicit bottoms, a `rank` (group or
formation) and a `parent` for each row. Groups overlap the formations they contain -
for example `BAAT GP.` (the Brent Group, 2841-2958 m) contains the Ness and Etive
formations - so a package built from every row would describe the same rock twice.
Select one level with `tops_options={"rank": "formation"}`; do not try to infer the
hierarchy by deleting nested rows.

In [3]:
formations = pp.read_formations(FORMATION_SOURCE)
display(formations[["formation_name", "top_depth", "bottom_depth", "rank", "parent"]])
assert set(formations["depth_unit"]) == {"m"}
assert {"group", "formation"}.issubset(set(formations["rank"]))
print("Selected formation intervals:", int((formations["rank"] == "formation").sum()))

,formation_name,top_depth,bottom_depth,rank,parent
0,NORDLAND GP.,181.0,726.0,group,
1,Utsira Fm.,726.0,879.0,formation,
2,HORDALAND GP.,879.0,1908.0,group,
3,ROGALAND GP.,1908.0,2203.0,group,
4,Balder Fm.,1908.0,2023.0,formation,ROGALAND GP.
5,Sele Fm.,2023.0,2123.0,formation,ROGALAND GP.
6,Lista Fm.,2123.0,2203.0,formation,ROGALAND GP.
7,SHETLAND GP.,2203.0,2408.0,group,
8,CROMER KNOLL GP.,2408.0,2452.0,group,
9,VIKING GP.,2452.0,2841.0,group,


Selected formation intervals: 12


## 4. Generate bounded, category-first context

`pp.export_context` with a package `layout` reads every LAS in the folder, applies
optional geology inputs, runs the workflow, groups the results by formation and lithology,
and validates every file before publishing the new directory. Category documents contain
the computed context directly, without replay or a wrapper object. A failed or oversized
export raises an exception without leaving a partial destination.

In [4]:
results = pp.export_context(
    LOGS_DIR, OUTPUT_DIR,
    layout="category_formation_lithology",
    tops_file=FORMATION_SOURCE,
    lithology_file=LITHOLOGY_SOURCE,
    tops_options={"rank": "formation"},
    minimum_group_thickness=MINIMUM_GROUP_THICKNESS,
    max_bytes=MAX_BYTES,
    max_total_bytes=50_000_000,
)
print("Generated records:", len(results))
assert results and all(item.success for item in results)

[repository]/src/petrophys/preprocessing/standardization/curve_standardizer.py:393: UserWarning: SWC: inferred percent core saturation despite unit 'frac'; divided by 100 to v/v. Verify source units.
  standardize_core_saturation(new_curve, inplace=True)
[repository]/src/petrophys/preprocessing/standardization/curve_standardizer.py:393: UserWarning: SOC: inferred percent core saturation despite unit 'frac'; divided by 100 to v/v. Verify source units.
  standardize_core_saturation(new_curve, inplace=True)


Generated records: 828


## 5. Summarize and review warnings

The API has already checked for failures. Still read the warnings before interpreting
the exported statistics; successful serialization is not scientific validation. For
example, the bundled 30/6-5 LAS header carries its own `RW` value (0.7 ohm.m, with no
reference temperature). The workflow records that source and its documented temperature
assumption in the trace so it can be reviewed.

In [5]:
context_files = sorted({Path(item.json_file) for item in results})
category_counts = Counter(path.relative_to(OUTPUT_DIR).parts[0] for path in context_files)
summary = pd.DataFrame({
    "category": list(category_counts),
    "json_files": list(category_counts.values()),
}).sort_values("category", ignore_index=True)
display(summary)
warnings = sorted({warning for item in results for warning in item.warnings})
print("Export warnings:", len(warnings))
for warning in warnings[:5]:
    print(warning)
assert all(path.stat().st_size <= MAX_BYTES for path in context_files)
print("JSON context files:", len(context_files))

,category,json_files
0,hydrocarbon,138
1,lithology,138
2,numerical,138
3,permeability,138
4,porosity,138
5,saturation,138


Export warnings: 4
Auto lithology resolved via external_lithology (93% of 22146 samples): lithology=clastic
External lithology: unmapped labels [], 30 uncovered samples. Labels/gaps are preserved; unclassified samples use configured calculation fallbacks, not inferred rock labels.
Header parameter maps to rw=0.7 but the config sets it explicitly; config value kept.
Marl uses the limestone matrix and carbonate cutoff family. This is a routing convention, not a calibrated marl model; check clay correction and supply project-specific cutoffs.
JSON context files: 828


In [6]:
# Count files per formation instead of listing hundreds of paths.
relative = [path.relative_to(OUTPUT_DIR) for path in context_files]
print("Layout : <category>/<well>/<formation>/<formation>_<lithology group>_<category>.json")
print("Example:", relative[0].as_posix())
per_formation = Counter(path.parts[2] for path in relative)
display(pd.Series(per_formation, name="json_files").sort_index().to_frame())

Layout : <category>/<well>/<formation>/<formation>_<lithology group>_<category>.json
Example: hydrocarbon/30_6-5/AMUNDSEN FM/AMUNDSEN FM._SILT_GROUP_2_hydrocarbon.json


,json_files
AMUNDSEN FM,18
BALDER FM,72
COOK FM,48
DRAKE FM,78
DRAUPNE FM,24
ETIVE FM,102
HEATHER FM,126
LISTA FM,36
NESS FM,18
SELE FM,42


## 6. Inspect one context document

Choose the category that matches the question being asked. Here we open the hydrocarbon
context of an Etive Formation (Brent reservoir) group and show only high-signal metadata
from the complete document.

In [7]:
hydrocarbon_files = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "hydrocarbon"
    and path.parent.name.lower().startswith("etive")
]
context_path = hydrocarbon_files[0] if hydrocarbon_files else context_files[0]
document = json.loads(context_path.read_text(encoding="utf-8"))
analysis = document["analysis_results"]
overview = pd.Series({
    "file": context_path.name,
    "well": analysis["well_name"],
    "formation": analysis["formation_name"],
    "lithology_group": analysis["lithology group"],
    "lithology": analysis["lithology"],
    "input_log_count": len(analysis["input_logs_present"]),
    "derived_log_count": len(analysis["derived_logs_computed"]),
    "raw_traces_included": document["include_raw_traces_in_context"],
})
display(overview.to_frame("value"))

,value
file,ETIVE FM._CROSSBEDDED SAND_GROUP_1_hydrocarbon...
well,30_6-5
formation,ETIVE FM.
lithology_group,CROSSBEDDED SAND_GROUP_1
lithology,CROSSBEDDED SAND
input_log_count,13
derived_log_count,31
raw_traces_included,False


In [8]:
# Basic schema and provenance checks before an LLM sees the document.
required_top_level = {
    "formation_info",
    "analysis_results",
    "raw_log_traces_in_segment",
    "include_raw_traces_in_context",
}
required_analysis = {
    "well_name",
    "formation_name",
    "statistics",
    "analysis_summary",
    "data_completeness",
    "input_logs_present",
    "derived_logs_computed",
    "BADHOLE_QC_CHECK",
}

assert required_top_level.issubset(document)
assert required_analysis.issubset(analysis)
assert analysis["well_name"]
assert document["formation_info"]["thickness"] >= 0
print("Schema and provenance checks passed.")

Schema and provenance checks passed.


### Scientific review, beyond schema

Inspect a known interval and the assumptions used to calculate it. The following numerical checks only verify valid ranges and the explicit Rw basis. They do not establish hydrocarbons, validate the chosen calibration, or justify shale/overburden pay flags. Review clay-volume methods, bad-hole QC, saturation-model assumptions, and supplied geology before treating cutoff screening as pay.

In [9]:
etive_sand = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "saturation"
    and any("etive" in part.lower() for part in path.parts)
    and "SAND" in path.name
]
assert etive_sand, "No Etive sand context found; review input selection"
probe = json.loads(etive_sand[0].read_text(encoding="utf-8"))
stats = probe["analysis_results"]["statistics"]
sw_mean = stats["SW_overall"]["arithmetic_mean"]
rw_mean = stats["RW_overall"]["arithmetic_mean"]
print(f"{etive_sand[0].name}")
print(f"SW mean: {sw_mean:.3f}; corrected Rw mean: {rw_mean:.3f} ohm.m")
assert 0 <= sw_mean <= 1
assert rw_mean > 0
print("Numerical checks passed; reservoir calibration still requires review.")

ETIVE FM._CROSSBEDDED SAND_GROUP_1_saturation.json
SW mean: 0.339; corrected Rw mean: 0.072 ohm.m
Numerical checks passed; reservoir calibration still requires review.


## 7. Build a model-neutral LLM request

Keep measured inputs, computed outputs, warnings, and missing-data statements intact. Ask
the model to distinguish observation from inference and to avoid inventing values. The
object below can be serialized for any LLM SDK or stored in a retrieval index.

In [10]:
llm_request = {
    "task": "Summarize the interval's measured evidence, calculations and limitations.",
    "instructions": [
        "Use only the supplied context.",
        "Separate measurements from calculated interpretations.",
        "Do not infer missing values or equate cutoff screening with confirmed pay.",
        "Report well, formation, lithology group and depth interval.",
        "Retain QC warnings and state which assumptions need calibration.",
    ],
    "petrophysical_context": document,
}
serialized_request = json.dumps(llm_request, ensure_ascii=False)
print("Request UTF-8 bytes:", len(serialized_request.encode("utf-8")))
print("Prepared locally; no LLM or network call was made.")
run_directory.cleanup()

Request UTF-8 bytes: 21491
Prepared locally; no LLM or network call was made.


## CLI equivalent

```bash
petrophys-context well.las context.json --formations formation_intervals.csv --lithology lithology_intervals.csv
petrophys-context logs outputs/context_run_01 --layout category_formation_lithology --formations formation_intervals.csv --lithology lithology_intervals.csv --tops-rank formation
```

Remove flags for optional inputs that are unavailable. Supported Rw, temperature and
Archie parameters come from LAS metadata; use `--lithology-domain carbonate` when the
broad mineral-solving domain is not stored in the LAS. The command always computes
context from the LAS and supplied geology tables. It never reads or copies an earlier
context package.

## Pitfalls

- **Pass the result, not the raw well.** `pp.build_context(well)` only summarizes what
  the well already holds; without an interpretation there are no calculated curves and
  no trace.
- **State Rw with its temperature.** Store `RW` and `RWT` together in LAS metadata,
  or in Python pass `rw` with `rw_meas_temp_f`. Otherwise the library documents its
  reference-temperature assumption.
- **Choose one formation level.** Mixing groups and formations describes the same rock
  twice; use `tops_options={"rank": "formation"}` (CLI: `--tops-rank formation`).
- **Package directories are never overwritten.** Export into a new directory; an
  existing one raises `FileExistsError`.
- **`minimum_group_thickness` is in the LAS depth unit** - 2.0 means 2 m for a metric
  LAS and 2 ft for a foot-based one.
- **Serialization is not validation.** Schema checks prove the files are well formed,
  not that the calibration, clay model or pay flags are right.

## Recap

- `pp.build_context(result)` and `pp.export_context(result, path)` work without geology.
- The CLI always interprets LAS input before exporting context.
- Package exports use LAS metadata and accept optional geology tables.
- Exports are byte-bounded and package directories are never overwritten.
- Rw always travels with its measurement temperature.
- Warnings and physical calibration matter beyond JSON/schema checks.
- Context generation is model-neutral and makes no LLM request.

## Try it yourself

> **Exercise 1.** Export the same inputs with `layout="formation_lithology"` into a new
> directory. How do the file count and folder structure differ from the category layout?
>
> **Exercise 2.** Re-run the package export into a new directory with
> `max_bytes=5_000`. What exception do you get, and what is left on disk?
>
> **Exercise 3.** Drop `rw_reference_temperature` and `rw_temperature_unit` from the
> parameter row and export again. What does the Rw decision in the trace now say, and
> how does the Etive saturation mean change?

## What's next

Continue with **20_mcp_server.ipynb**. Instead of preparing context for a model, it lets
an LLM client call Petrophys directly through the Model Context Protocol: load wells,
interpret, explain steps, summarize zones and plot - with every number coming from the
same traced workflow.